# Nova AI — دمج النماذج (Mergekit) + تدريب خفيف (Unsloth)

**التشغيل من الهاتف (Kaggle، وليس Colab):** دمج نموذجين بحجم 7B معاً
يحتاج فعلياً حوالي 30GB من الذاكرة — أثبتنا هذا مباشرة بالتجربة على
Colab المجاني (~12.7GB CPU RAM أو 15GB GPU VRAM فقط)، فتوقف الدمج
بنفاد الذاكرة في كل مرة. **Kaggle Notebooks** (مجاني بالكامل أيضاً)
يوفر **29GB من CPU RAM** — أقرب بكثير لما نحتاجه فعلياً.

خطوات الفتح من الهاتف:
1. أنشئ حساباً مجانياً على kaggle.com (يمكن مباشرة بحساب Google)
2. من صفحتك الرئيسية: **Create** -> **New Notebook**
3. من قائمة **File** أعلى الدفتر الجديد: **Import Notebook** -> تبويب
   **GitHub** -> الصق رابط هذا الملف مباشرة:
   `https://github.com/<owner>/<repo>/blob/<branch>/ai-system/colab/merge_and_finetune.ipynb`
4. من اللوحة اليمنى **Notebook options** -> **Accelerator** اختر **GPU T4 x2** (أو أي T4 متاح) — Kaggle يطلب تفعيل رقم هاتفك مرة واحدة فقط لتفعيل هذا الخيار (إجراء أمان معتاد منهم، مجاني)
5. شغّل كل خلية بالترتيب بزر ▷، تماماً كما في Colab

**ما يفعله هذا الدفتر:**
1. يدمج نموذجين مفتوحين متوسطي الحجم (Qwen2.5-Coder-7B-Instruct + Qwen2.5-7B-Instruct) عبر خوارزمية TIES في Mergekit — ينتج نموذجاً هجيناً واحداً يجمع قوة البرمجة مع قوة التحليل اللغوي.
2. (اختياري) يعمل تدريباً خفيفاً LoRA عبر Unsloth على بيانات خاصة بك.
3. يرفع الناتج إلى حسابك على Hugging Face Hub.
4. تضع اسم النموذج الناتج في `HF_SPECIALIST_MODEL_ID` داخل `ai-system/.env` — يصبح فوراً "الرأي المتخصص" الثالث في مجلس نماذج Nova للأسئلة البرمجية.

هذه الجلسة المجانية تنقطع بعد ساعات من الخمول — هذا متوقّع وطبيعي، هذا الدفتر مخصص للتشغيل الدوري (مرة كل أسبوع مثلاً) لإنتاج نسخة محسّنة، وليس لخدمة المستخدمين مباشرة (تلك مهمة Groq/Gemini الدائمين — انظر ai-system/README.md).

In [ ]:
# الخلية 1 — تثبيت الأدوات (يأخذ بضع دقائق أول مرة فقط)
!pip install -q mergekit huggingface_hub
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# الخلية 2 — تسجيل الدخول لحسابك على Hugging Face
# احصل على توكن (Write access) مجاناً من: https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# الخلية 3 — إعداد دمج النماذج (Mergekit، خوارزمية TIES)
# غيّر هذين السطرين إذا أردت تجربة نموذجين آخرين متوافقين بنفس البنية والحجم.
merge_config = """
models:
  - model: Qwen/Qwen2.5-Coder-7B-Instruct
    parameters:
      weight: 0.6
      density: 0.6
  - model: Qwen/Qwen2.5-7B-Instruct
    parameters:
      weight: 0.4
      density: 0.6
merge_method: ties
base_model: Qwen/Qwen2.5-7B-Instruct
parameters:
  normalize: true
dtype: float16
"""
with open("merge_config.yaml", "w") as f:
    f.write(merge_config)
print("تمت كتابة merge_config.yaml")

In [ ]:
# الخلية 4 — تنفيذ الدمج الفعلي، عبر واجهة Python الداخلية لـ mergekit
# (وليس أمر !mergekit-yaml كعملية طرفية منفصلة).
#
# 1) هناك خلل حقيقي داخل مكتبة mergekit نفسها — عدة أصناف داخل ملف
# mergekit/architecture/base.py (مثل ConfiguredModuleArchitecture و
# ConfiguredModelArchitecture) تفشل تلقائياً في "حل" أنواعها المرجعية
# (torch، PretrainedConfig، وغيرها) عند إنشائها لأول مرة. الحل: نعيد بناء
# كل الأصناف في هذا الملف يدوياً قبل الدمج.
#
# 2) دمج 7B+7B يحتاج فعلياً ~30GB ذاكرة — أثبتنا هذا بتجربة مباشرة على
# Colab (فشل على GPU 15GB وعلى CPU RAM ~12.7GB كليهما، بغض النظر عن أي
# ضبط لخيارات mergekit). لهذا ننفّذ هذه الخلية الآن على Kaggle الذي يوفر
# 29GB من CPU RAM — cuda=False لأن المتوفر هنا هو ذاكرة معالج وفيرة، لا
# حاجة لإقحام GPU (15-16GB) في عملية تحتاج أكثر من سعته أصلاً.
import inspect
import torch
import yaml
from pydantic import BaseModel
from transformers import PretrainedConfig
import mergekit.architecture.base as _mkbase
_ns = dict(vars(_mkbase))
_ns.update({"PretrainedConfig": PretrainedConfig, "torch": torch})
for _name, _obj in list(vars(_mkbase).items()):
    if inspect.isclass(_obj) and issubclass(_obj, BaseModel):
        try:
            _obj.model_rebuild(force=True, _types_namespace=_ns)
        except Exception as _e:
            print("تعذر إعادة بناء", _name, "-", _e)
from mergekit.config import MergeConfiguration
from mergekit.merge import run_merge
from mergekit.options import MergeOptions
config_source = open("merge_config.yaml", "r", encoding="utf-8").read()
parsed_config = MergeConfiguration.model_validate(yaml.safe_load(config_source))
merge_options = MergeOptions(cuda=False, low_cpu_memory=False, lazy_unpickle=True)
run_merge(parsed_config, out_path="./nova-merged-model", options=merge_options, config_source=config_source)
print("اكتمل الدمج بنجاح")

In [ ]:
# الخلية 5 — رفع النموذج المدموج إلى حسابك (غيّر your-username باسمك الفعلي)
from huggingface_hub import HfApi

HF_USERNAME = "your-username"  # <-- غيّر هذا
REPO_ID = f"{HF_USERNAME}/nova-coder-merge-7b"

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path="./nova-merged-model", repo_id=REPO_ID)
print(f"تم الرفع: https://huggingface.co/{REPO_ID}")
print("ضع هذا في HF_SPECIALIST_MODEL_ID داخل ai-system/.env:", REPO_ID)

---
## (اختياري) الخطوة الثانية: تدريب خفيف LoRA عبر Unsloth

شغّل الخلايا التالية فقط إذا كان لديك بيانات مخصصة (أسئلة/أجوبة حقيقية جمعتها من استخدام Nova) تريد تحسين النموذج المدموج عليها. استبدل `training_data` بأمثلتك الفعلية — كل عنصر هو زوج سؤال/إجابة.

In [ ]:
# الخلية 6 — تحميل النموذج المدموج عبر Unsloth للتدريب السريع (4-bit QLoRA)
#
# "ذاكرة إضافية فوق سعة الـ GPU" — تفعيل تلقائي بالكامل، بدون أي إعداد
# يدوي منك، ويعمل بنفس الطريقة على Kaggle كما كان على Colab (كود بايثون
# عام، لا علاقة له بالمنصة): يحاول الكود التحميل العادي داخل الـ GPU
# أولاً (الأسرع)، وفقط عندما يفشل فعلاً بسبب امتلاء الذاكرة (CUDA Out Of
# Memory) — أي نموذج أكبر من سعة الـ GPU المتاحة — يعيد المحاولة
# تلقائياً موزّعاً الطبقات بين الـ GPU وذاكرة CPU RAM والقرص (Offloading)،
# أبطأ لكنه ينجح بدل أن يتوقف بخطأ. النموذج الحالي (7B بصيغة 4-bit ≈
# 5GB) لن يحتاج هذا إطلاقاً وسيمر مباشرة من المحاولة الأولى.
import torch
from unsloth import FastLanguageModel

MAX_GPU_MEMORY = "15GiB"   # هامش أمان عن حد T4 على Kaggle (~16GB)
MAX_CPU_MEMORY = "28GiB"   # هامش أمان عن حد ذاكرة Kaggle المجانية (~29GB) — تمدّد سعة الـ GPU عند الحاجة

def _load_model(offload: bool):
    extra_kwargs = {}
    if offload:
        extra_kwargs = {
            "device_map": "auto",
            "max_memory": {0: MAX_GPU_MEMORY, "cpu": MAX_CPU_MEMORY},
            "offload_folder": "./offload",
        }
    return FastLanguageModel.from_pretrained(
        model_name=REPO_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        **extra_kwargs,
    )

try:
    model, tokenizer = _load_model(offload=False)
    print("تم التحميل مباشرة داخل الـ GPU — النموذج يتسع ضمن الذاكرة المتوفرة.")
except torch.cuda.OutOfMemoryError:
    print("النموذج أكبر من سعة الـ GPU — تفعيل تمديد الذاكرة (CPU/Disk Offload) تلقائياً...")
    torch.cuda.empty_cache()
    model, tokenizer = _load_model(offload=True)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

In [ ]:
# الخلية 7 — بياناتك: استبدل هذه الأمثلة بأسئلة/أجوبة حقيقية جمعتها
from datasets import Dataset

training_data = [
    {"question": "مثال سؤال حقيقي جمعته من مستخدمي Nova", "answer": "مثال الإجابة المثالية له"},
]

def format_example(ex):
    return {"text": f"### سؤال:\n{ex['question']}\n### إجابة:\n{ex['answer']}"}

dataset = Dataset.from_list(training_data).map(format_example)

In [ ]:
# الخلية 8 — التدريب الفعلي (سريع، دقائق فقط لعشرات/مئات الأمثلة)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        learning_rate=2e-4,
        output_dir="./nova-lora-out",
        logging_steps=1,
    ),
)
trainer.train()

In [ ]:
# الخلية 9 — رفع النسخة المُحسَّنة (تستبدل نفس REPO_ID أو أنشئ اسماً جديداً)
model.push_to_hub_merged(REPO_ID, tokenizer, save_method="merged_16bit")
print(f"تم تحديث النموذج على: https://huggingface.co/{REPO_ID}")